In [ ]:
"""
cohort_scoring.py - shared cohort scorecard maths (pure pandas, no I/O).
Customer Health & Sentiment Scoring Multi-Agent AI Solution

Used by:
  eda_data_understanding_v6.py  Step 12 / 12A  (exploratory scorecard on the EDA cm table)
  step3d_build_cohort.py                       (production scorecard on the Step 3c feature table)
Both scripts therefore rank candidates with identical logic.

Change log (see decisions.md): D10 cohort_score_ungated added so EDA Step 12A can compare rules without
re-implementing the maths; deterministic tie-break on dimension/segment.
"""

## 1. Imports and required scorecard columns

In [ ]:
from __future__ import annotations

import pandas as pd

REQUIRED_COLUMNS = ("dimension", "segment", "n_core_customers", "coverage_pct", "avg_active_months_history",
                    "survey_label_pct", "interaction_label_pct", "avg_spend_pctile", "std_spend_pctile",
                    "risk_signal_pct", "max_mom_swing_pct")

## 2. Stability: largest month-on-month swing in core customer-months

In [ ]:
def max_mom_swing(monthly: pd.DataFrame) -> pd.DataFrame:
    """monthly: columns segment, month, n (core customer-months). Returns segment, max_mom_swing_pct."""
    m = monthly[monthly["n"] > 0].sort_values("month")
    if m.empty:
        return pd.DataFrame(columns=["segment", "max_mom_swing_pct"])
    return (m.groupby("segment")["n"].apply(lambda s: (s.pct_change().abs() * 100).max())
            .rename("max_mom_swing_pct").reset_index())

## 3. Candidate ranking

In [ ]:
def _weighted_score(cand: pd.DataFrame, mask: pd.Series, weights: dict) -> pd.Series:
    """Weighted sum of percentile ranks, ranks computed among rows in `mask` only (others rank 0)."""
    def rank(col: str, asc: bool = True) -> pd.Series:
        r = pd.Series(0.0, index=cand.index)
        r[mask] = cand.loc[mask, col].rank(pct=True, ascending=asc).fillna(0.0)   # NaN metric -> lowest rank
        return r

    parts = {
        "value": rank("avg_spend_pctile") * 0.5 + rank("risk_signal_pct") * 0.5,
        "coverage": rank("coverage_pct"),
        "labels": rank("survey_label_pct") * 0.5 + rank("interaction_label_pct") * 0.5,
        "sample": rank("n_core_customers"),
        "history": rank("avg_active_months_history"),
        "stability": rank("max_mom_swing_pct", asc=False),
        "homogeneity": rank("std_spend_pctile", asc=False),
    }
    for k, v in parts.items():
        cand[f"s_{k}"] = v
    return sum(cand[f"s_{k}"] * w for k, w in weights.items()).round(3)


def score_candidates(sc: pd.DataFrame, min_customers: int, weights: dict) -> pd.DataFrame:
    """Rank candidate cohorts (all rows except dimension == 'ALL').

    cohort_score          ranks computed ONLY among candidates that pass the sample-size gate, so a tiny segment
                          can never shift the ranks of eligible ones; failing candidates get 0.
    cohort_score_ungated  same weights, ranks over ALL candidates (diagnostic - used by EDA Step 12A to compare
                          rules even when a candidate fails the gate). Never used to select the cohort.
    """
    missing = [c for c in REQUIRED_COLUMNS if c not in sc.columns]
    if missing:
        raise KeyError(f"scorecard is missing columns: {missing}")
    cand = sc[sc["dimension"] != "ALL"].copy()
    cand["passes_sample_gate"] = cand["n_core_customers"] >= min_customers
    ok = cand["passes_sample_gate"]

    cand["cohort_score_ungated"] = _weighted_score(cand, pd.Series(True, index=cand.index), weights)
    cand["cohort_score"] = _weighted_score(cand, ok, weights)          # s_* columns = gated components
    cand.loc[~ok, "cohort_score"] = 0.0
    return (cand.sort_values(["cohort_score", "cohort_score_ungated", "n_core_customers", "dimension", "segment"],
                             ascending=[False, False, False, True, True])
            .reset_index(drop=True))